# BoltzGen input cases

In [ ]:
import time

import biotite.structure as struc
import torch

from evedesign.constants import MASK
from evedesign.models.boltzgen import BoltzGenGenerator
from evedesign.system import (
    Interaction, Ligand, Protein, SecondaryStructure, System,
)
from evedesign.structure import Structure, StructureFile

assert torch.cuda.is_available(), "boltzgen has no CPU path"
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

In [ ]:
NUM_DESIGNS = 50
BUDGET = 10
RESULTS = []
DESIGNS = {}


def run_case(name, system, protocol="protein-anything", fixed_pos=None,
             entities=(1,)):
    """entities defaults to the binder: every case is [target, binder],
    and the target is held fixed."""
    t0 = time.perf_counter()
    row = {"case": name, "designs": 0, "seq": "", "score": None, "error": ""}
    try:
        gen = BoltzGenGenerator(
            protocol=protocol,
            device="cuda",
            num_devices=2,
            budget=BUDGET,
            skip_inverse_folding=True,
            keep_tmp_dir=True,
        ).build(system)
        designs = gen.generate(num_designs=NUM_DESIGNS, fixed_pos=fixed_pos,
                               entities=entities)
        DESIGNS[name] = designs
        row["designs"] = len(designs)
        if designs:
            binder = designs[0][-1]
            if binder.rep is not None:
                row["seq"] = "".join(binder.rep)[:24]
            row["score"] = designs[0].score
    except Exception as exc:
        row["error"] = f"{type(exc).__name__}: {exc}"[:140]
    row["min"] = round((time.perf_counter() - t0) / 60, 1)
    RESULTS.append(row)
    print(row)
    return row

In [ ]:
chain = StructureFile.from_id("1G13").get_model().get_chain("A")
TARGET_STRUCT = Structure(chain.atom_array[struc.filter_amino_acids(chain.atom_array)])
TARGET_SEQ = "".join(TARGET_STRUCT.res_df().res_name_oneletter)


def target(**kwargs):
    """1G13 chain A as a structural target, matching boltzgen's examples,
    which pass targets as file: entries rather than bare sequences."""
    return Protein(rep=TARGET_SEQ, id="target",
                   structures={"1g13": TARGET_STRUCT}, **kwargs)




print(f"target: {len(TARGET_SEQ)} aa")

## 1. Baseline: sequence target, variable-length binder

In [ ]:
run_case("baseline", System([
    target(),
    Protein(rep=None, min_length=60, max_length=80, id="binder"),
]))


## 2. Motif scaffolding via fixed_pos

In [ ]:
MOTIF = "GGGGWKQLADQLYRAG"

run_case("motif", System([
    target(),
    Protein(rep=MOTIF, min_length=len(MOTIF), max_length=len(MOTIF),
            id="binder"),
]), fixed_pos={1: [5, 6, 7, 8, 9]})


## 2b. Mask symbols in rep mark designable positions

Mask symbols (`*`) mark positions that must be designed; everything else in the rep is held. Masks are fixed-length by convention, so 6 stars are 6 designed residues at defined sequence positions.

The rep below emits `GGGG6WKQLADQLYRAG`, the same letters-fixed/digits-designed form as case 2, but expressed on the entity itself rather than through `fixed_pos`.

In [ ]:
MASKED = "GGGG" + MASK * 6 + "WKQLADQLYRAG"

run_case("mask", System([
    target(),
    Protein(rep=MASKED, id="binder"),
]))

## 2c. fixed_pos is numbered from Entity.first_index

Position specs follow the entity's own numbering, not BoltzGen's 1-based chain index. Here the binder starts at residue 101, so `fixed_pos` names 105-109 to hold the same five residues that case 2 held with 5-9. Both should produce the same spec.

In [ ]:
run_case("motif_first_index", System([
    target(),
    Protein(rep=MOTIF, first_index=101, id="binder"),
]), fixed_pos={1: [105, 106, 107, 108, 109]})

## 3. Secondary structure on the designed chain

In [ ]:
run_case("secondary_structure", System([
    target(),
    Protein(rep=None, min_length=60, max_length=60, id="binder",
            secondary_structure=[
                SecondaryStructure(pos=p, type="H") for p in range(10, 31)
            ]),
]))


## 4. Binding site on the target

In [ ]:
run_case("binding_types", System([
    target(interactions=[Interaction(id="site", pos=list(range(40, 61)))]),
    Protein(rep=None, min_length=60, max_length=80, id="binder"),
]))


## 5. Cyclic binder

In [ ]:
run_case("cyclic", System([
    target(),
    Protein(rep=None, min_length=12, max_length=12, id="binder", cyclic=True),
]), protocol="peptide-anything")


## 6. Homo-oligomer target

In [ ]:
run_case("homo_oligomer", System([
    Protein(rep=TARGET_SEQ, id="target", copies=2),
    Protein(rep=None, min_length=60, max_length=80, id="binder"),
]))

## 7. Ligand target

In [ ]:
run_case("ligand", System([
    Ligand(rep="ATP", ligand_rep_type="ccd", id="lig"),
    Protein(rep=None, min_length=60, max_length=80, id="binder"),
]), protocol="protein-small_molecule")

## 8. Sequence-only target

In [ ]:
run_case("sequence_target", System([
    Protein(rep=TARGET_SEQ, id="target"),
    Protein(rep=None, min_length=60, max_length=80, id="binder"),
]))


## Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(RESULTS)
print(df.to_string(index=False))
print()
print(f"{(df['designs'] > 0).sum()}/{len(df)} cases produced designs")


## Verify each case had the intended effect

A case producing designs only shows BoltzGen accepted the spec. These checks read the returned sequences and structures back to confirm the conditioning actually took: held residues survived, lengths were honoured, and requested secondary structure appears in the output backbone.

In [ ]:
def binder(name):
    """First design's binder entity instance, or None if the case failed."""
    designs = DESIGNS.get(name)
    return designs[0][-1] if designs else None


def check(name, label, fn):
    b = binder(name)
    if b is None:
        print(f"  SKIP  {name}: no designs")
        return
    ok = fn("".join(b.rep), b.models["model_0"].res_df())
    print(f"  {'PASS' if ok else 'FAIL'}  {name}: {label}")


# Held residues survive design, and masks do not change the length
check("motif", "fixed_pos residues kept",
      lambda seq, df: seq[4:9] == MOTIF[4:9])
check("motif_first_index", "first_index offset holds the same residues",
      lambda seq, df: seq[4:9] == MOTIF[4:9])
check("mask", "unmasked residues kept, length unchanged",
      lambda seq, df: len(seq) == len(MASKED)
      and seq[:4] == MASKED[:4] and seq[10:] == MASKED[10:])

# Lengths honoured
check("cyclic", "12 residues",
      lambda seq, df: len(seq) == 12)
check("baseline", "length within min..max",
      lambda seq, df: 60 <= len(seq) <= 80)

# Requested conditioning visible in the output backbone
check("secondary_structure", "helix over positions 10..30",
      lambda seq, df: (df["sse"][9:30] == "H").mean() > 0.5)